# JetRacer Local GPU Trainer

このノートブックは、ローカルのGPUマシンを利用してJetRacerのAIモデルを学習するためのツールです。

**主な機能:**
- **インタラクティブUI:** ボタンやスライダーで直感的に操作できます。
- **データセット管理:** zipファイルをアップロードして展開したり、既存のデータセットを再利用したりできます。
- **学習結果のCSV出力:** 学習の進捗をCSVファイルに保存し、後からグラフ化や分析が可能です。
- **リアルタイム進捗表示:** 学習状況や評価動画をリアルタイムで確認できます。

### **ステップ0: 準備**

最初に、必要なライブラリがインストールされていることを確認し、補助スクリプトをダウンロードします。

In [ ]:
import torch
import os

# GPUが利用可能かを確認
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPUが利用可能です: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('⚠️ GPUが見つかりません。CPUで学習を実行します。')

# 補助スクリプトが存在しない場合のみダウンロード
if not os.path.exists('utils.py'):
    print('Downloading utils.py...')
    !wget -q https://raw.githubusercontent.com/NVIDIA-AI-IOT/jetracer/master/notebooks/utils.py

if not os.path.exists('xy_dataset.py'):
    print('Downloading xy_dataset.py...')
    !wget -q https://raw.githubusercontent.com/NVIDIA-AI-IOT/jetracer/master/notebooks/xy_dataset.py

print("✅ 準備が完了しました。")

### **ステップ1: データセットの準備**

このノートブックと同じ階層に`datasets`フォルダが作成されます。ここにデータセットが保存されます。

- **既存データセットの読込:** 下のセルを実行すると、`datasets`フォルダ内にあるデータセットが自動でリストに表示されます。
- **新規データセットの追加:** 「Upload Datasets (.zip)」ボタンを使い、JetRacerで収集したzipファイルをアップロードしてください。アップロードが完了すると、自動で展開されリストが更新されます。

In [ ]:
import ipywidgets as widgets
import zipfile
import io
import os
import shutil
from IPython.display import display, clear_output
import numpy as np

# --- グローバル設定 ---
DATASET_DIR = 'datasets'
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)

# --- UIウィジェット ---
uploader = widgets.FileUpload(
    accept='.zip',
    multiple=True,  # 複数ファイルのアップロードを許可
    description='Upload Datasets (.zip)',
    button_style='primary',
    icon='upload'
)
upload_output = widgets.Output() # ログや処理状況を表示するエリア
dataset_list_widget = widgets.SelectMultiple(
    options=[],
    description='学習用データセット:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='150px')
)

# --- 関数 ---
def find_xy_path(root_dir):
    """指定されたディレクトリ内を再帰的に検索し、'xy'フォルダを含む親ディレクトリのパスを返す"""
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if 'xy' in dirnames:
            return dirpath
    return None

def scan_and_update_dataset_list():
    """DATASET_DIRをスキャンして、有効なデータセットをリストに反映する"""
    found_datasets = []
    if not os.path.exists(DATASET_DIR):
        return
    # DATASET_DIR直下のディレクトリのみをスキャン対象とする
    for item in os.listdir(DATASET_DIR):
        item_path = os.path.join(DATASET_DIR, item)
        if os.path.isdir(item_path):
            final_path = find_xy_path(item_path)
            if final_path:
                found_datasets.append(final_path)
    
    dataset_list_widget.options = sorted(found_datasets)
    with upload_output:
        clear_output(wait=True)
        print(f"Found {len(found_datasets)} existing datasets.")

def on_upload_change(change):
    """ファイルがアップロードされたときの処理"""
    uploaded_files = uploader.value
    if not uploaded_files:
        return
        
    with upload_output:
        clear_output(wait=True)
        print("アップロード処理中...")

        # uploaded_filesは辞書のタプル
        for file_info in uploaded_files:
            name = file_info['name']
            content = file_info['content']
            try:
                with zipfile.ZipFile(io.BytesIO(content), 'r') as zf:
                    # zipファイル名から拡張子を除いた名前でディレクトリを作成
                    extract_base_path = os.path.join(DATASET_DIR, os.path.splitext(name)[0])
                    if os.path.exists(extract_base_path):
                        print(f"ℹ️ '{extract_base_path}' は既に存在するため、上書きします。")
                        shutil.rmtree(extract_base_path)
                    os.makedirs(extract_base_path)
                    
                    zf.extractall(extract_base_path)
                    print(f"✅ '{name}' を '{extract_base_path}' に展開しました。")

            except Exception as e:
                print(f"❌ '{name}' の処理中にエラーが発生しました: {e}")
        
        # 処理が終わったらアップローダーをリセット
        # これをしないと、同じファイルを再度アップロードしてもイベントが発生しない
        # uploader.value.clear()はAttributeErrorを出すことがあるのでtry-exceptで囲む
        try:
            uploader.value.clear()
        except Exception:
            pass
        
        # データセットリストを再スキャンして更新
        scan_and_update_dataset_list()
        print("\nデータセットリストを更新しました。")

# --- 初期化と表示 ---
uploader.observe(on_upload_change, names='value')
display(widgets.VBox([
    uploader, 
    upload_output, 
    dataset_list_widget
]))

# 実行時に既存のデータセットをスキャン
scan_and_update_dataset_list()

### **ステップ2: モデルの学習**

上のリストから学習に使用したいデータセットを選択（複数選択可: `Ctrl` or `Shift` + クリック）し、エポック数などを設定して「学習開始」ボタンを押してください。

学習が完了すると、最も性能の良かったモデル `best_model.pth` と、学習記録 `learning_log.csv` がこのノートブックと同じ階層に保存されます。

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import ConcatDataset, DataLoader, random_split
import torch.optim as optim
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
import ipywidgets as widgets

# このセルを実行する前に、xy_dataset.pyが同じディレクトリにあることを確認してください
from xy_dataset import XYDataset

# --- モデル定義と学習関数 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = None

# 【修正1】出力次元のデフォルトを4に変更 (xy:2 + speed:2)
def get_model(output_dim=4):
    model = torchvision.models.resnet18(weights='DEFAULT')
    model.fc = torch.nn.Linear(512, output_dim)
    return model.to(device)

def train_model(b):
    global model
    train_button.disabled = True
    train_log_widget.value = "学習準備中...\n"
    with graph_output_live:
        clear_output()

    # データセットの準備
    selected_datasets = dataset_list_widget.value
    if not selected_datasets:
        train_log_widget.value = "エラー: 学習するデータセットが選択されていません。"
        train_button.disabled = False
        return
        
    all_datasets = []
    train_log_widget.value += "データセットを読み込んでいます...\n"
    for path in selected_datasets:
        try:
            # カテゴリを 'xy' と 'speed' の両方に設定
            dataset = XYDataset(path, 
                categories=['xy', 'speed'], 
                transform=transforms.Compose([
                    transforms.Lambda(lambda x: x.crop((0, 75, 224, 224))), # 1. ノイズ除去 (PIL Crop)
                    transforms.Resize((224, 224)),       # 2. 形状適合
                    transforms.ColorJitter(0.2, 0.2, 0.2, 0.2), # 3. データ拡張
                    transforms.ToTensor(),               # 4. Tensor化
                    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # 5. 正規化
                ]), random_hflip=True)
            all_datasets.append(dataset)
            train_log_widget.value += f"- '{path}' ({len(dataset)}件) 読み込み完了\n"
        except Exception as e:
             train_log_widget.value += f"- 警告: '{path}' の読み込み失敗: {e}\n"
    
    if not all_datasets:
        train_log_widget.value += "\nエラー: 有効なデータセットがありません。"
        train_button.disabled = False
        return

    full_dataset = ConcatDataset(all_datasets)
    
    # データの分割
    test_percent = test_split_widget.value / 100.0
    test_size = int(len(full_dataset) * test_percent)
    train_size = len(full_dataset) - test_size
    
    if train_size == 0 or test_size == 0:
         # データが少なすぎる場合の安全策
        train_size = len(full_dataset)
        test_size = 0
        train_dataset = full_dataset
        test_dataset = [] 
        train_log_widget.value += "\n警告: データセットが小さすぎるため、すべて学習用に使用します。\n"
    else:
        train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_widget.value, shuffle=True, num_workers=0)
    if test_size > 0:
        test_loader = DataLoader(test_dataset, batch_size=batch_widget.value, shuffle=False, num_workers=0)
    else:
        test_loader = None
    
    # モデルとオプティマイザの初期化 (output_dim=4)
    model = get_model(output_dim=4)
    optimizer = optim.Adam(model.parameters(), lr=lr_widget.value)
    
    best_loss = float('inf')
    epochs = epochs_widget.value
    log_data = []

    train_log_widget.value += f"学習を開始します (Total: {len(full_dataset)}, Train: {train_size}, Test: {test_size})\n"
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        # 【修正2】category_idxを使用する学習ループに変更 (15_train.ipynb準拠)
        for images, category_idx, xy in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images = images.to(device)
            xy = xy.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            # 15_train.ipynb と同じ損失計算ロジック
            loss = 0.0
            for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                # cat_idx=0 (xy) なら outputs[0:2], cat_idx=1 (speed) なら outputs[2:4] を使用
                loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx+2] - xy[batch_idx])**2)
            loss /= len(category_idx)
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        avg_test_loss = 0.0
        
        # テスト評価 (データがある場合のみ)
        if test_loader:
            model.eval()
            test_loss = 0.0
            with torch.no_grad():
                for images, category_idx, xy in tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [Test]"):
                    images = images.to(device)
                    xy = xy.to(device)
                    outputs = model(images)
                    
                    # テスト時も同じ損失計算ロジック
                    loss = 0.0
                    for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                        loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx+2] - xy[batch_idx])**2)
                    loss /= len(category_idx)
                    
                    test_loss += loss.item()
            avg_test_loss = test_loss / len(test_loader)

        log_data.append({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'test_loss': avg_test_loss
        })
        
        train_log_widget.value += f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.5f}, Test Loss: {avg_test_loss:.5f}\n"
        train_progress_widget.value = (epoch + 1) / epochs
        
        # ベストモデルの保存判定
        current_val_loss = avg_test_loss if test_loader else avg_train_loss
        if current_val_loss < best_loss:
            best_loss = current_val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            train_log_widget.value += f"  -> ✨ New best model saved with loss: {best_loss:.5f}\n"
        
        # グラフ更新
        with graph_output_live:
            clear_output(wait=True)
            df_live = pd.DataFrame(log_data)
            plt.style.use('seaborn-v0_8-whitegrid')
            plt.figure(figsize=(10, 6))
            plt.plot(df_live['epoch'], df_live['train_loss'], 'o-', label='Train Loss')
            if test_loader:
                plt.plot(df_live['epoch'], df_live['test_loss'], 'o-', label='Test Loss')
            plt.title(f"Learning Curve (Train: {train_size} / Test: {test_size})")
            plt.xlabel('Epochs')
            plt.ylabel('Loss')
            plt.legend()
            plt.grid(True)
            plt.show()

    # CSVログの保存
    df = pd.DataFrame(log_data)
    df.to_csv('learning_log.csv', index=False)
    
    train_log_widget.value += "\n✅ 学習が完了しました。\n"
    train_log_widget.value += "'best_model.pth' と 'learning_log.csv' が保存されました。\n"
    train_button.disabled = False

# --- UIウィジェット (学習) ---
epochs_widget = widgets.IntText(description='Epochs:', value=30, layout=widgets.Layout(width='200px'))
batch_widget = widgets.IntText(description='Batch Size:', value=8, layout=widgets.Layout(width='200px'))
lr_widget = widgets.FloatText(description='Learning Rate:', value=1e-3, step=1e-4, layout=widgets.Layout(width='200px'), format='.0e')
test_split_widget = widgets.IntSlider(description='Test Split (%):', value=10, min=5, max=50, step=5, layout=widgets.Layout(width='300px'))
train_button = widgets.Button(description='学習開始', button_style='success', icon='rocket')
train_progress_widget = widgets.FloatProgress(min=0.0, max=1.0, description='Progress:')
train_log_widget = widgets.Textarea(layout=widgets.Layout(width='100%', height='250px'), description='Log:')
graph_output_live = widgets.Output() # リアルタイムグラフ表示用

# --- イベントリスナー (学習) ---
train_button.on_click(train_model)

# --- UI表示 (学習) ---
display(widgets.VBox([
    widgets.HBox([epochs_widget, batch_widget, lr_widget]),
    test_split_widget,
    train_button, 
    train_progress_widget, 
    train_log_widget,
    graph_output_live
]))

### **ステップ3: 学習曲線の確認**

ステップ2で保存された`learning_log.csv`ファイルを元に、学習の進捗をグラフで表示します。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- UIウィジェット ---
show_graph_button = widgets.Button(
    description="学習グラフを再表示",
    button_style='info'
)
graph_output = widgets.Output() # グラフ描画用の出力エリア

# --- 関数 ---
def show_learning_curve(b):
    with graph_output:
        clear_output(wait=True)
        log_file = 'learning_log.csv'
        if not os.path.exists(log_file):
            print(f"エラー: {log_file} が見つかりません。先にステップ2の学習を実行してください。")
            return
        
        try:
            df = pd.read_csv(log_file)
            
            plt.style.use('seaborn-v0_8-whitegrid')
            fig, ax = plt.subplots(figsize=(10, 6))
            
            ax.plot(df['epoch'], df['train_loss'], marker='o', linestyle='-', label='Train Loss')
            ax.plot(df['epoch'], df['test_loss'], marker='o', linestyle='-', label='Test Loss')
            
            ax.set_title('Learning Curve', fontsize=16)
            ax.set_xlabel('Epoch', fontsize=12)
            ax.set_ylabel('Loss', fontsize=12)
            ax.legend(fontsize=12)
            ax.grid(True)
            
            min_test_loss_epoch = df.loc[df['test_loss'].idxmin()]
            ax.annotate(f"Best Model\nEpoch: {int(min_test_loss_epoch['epoch'])}",
                        xy=(min_test_loss_epoch['epoch'], min_test_loss_epoch['test_loss']),
                        xytext=(min_test_loss_epoch['epoch'] + 1, min_test_loss_epoch['test_loss'] + 0.01),
                        arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8))

            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"グラフの描画中にエラーが発生しました: {e}")

# --- イベントリスナー ---
show_graph_button.on_click(show_learning_curve)

# --- UI表示 ---
display(widgets.VBox([show_graph_button, graph_output]))

### **ステップ4: 評価動画の作成**

学習したモデルの性能を視覚的に確認するための動画を作成します。

評価に使用するデータセット（学習に使っていないデータが望ましい）を選択し、「動画を作成」ボタンを押してください。新しい評価データはステップ1でアップロードできます。

In [ ]:
import cv2
import glob
from utils import preprocess
from IPython.display import HTML
from base64 import b64encode
import ipywidgets as widgets
import os
from tqdm.notebook import tqdm

# --- UIウィジェット (評価) ---
video_dataset_widget = widgets.Dropdown(options=dataset_list_widget.options, description='評価データセット:')
video_name_widget = widgets.Text(description='動画ファイル名:', value='evaluation_video.mp4')
create_video_button = widgets.Button(description='動画を作成', button_style='success', icon='video')
video_log_widget = widgets.Textarea(layout=widgets.Layout(width='100%', height='150px'), description='Log:')
video_player_widget = widgets.Output()

# --- 関数 (評価) ---
def create_video(b):
    create_video_button.disabled = True
    video_log_widget.value = "動画作成を開始します...\n"

    # 学習済みモデルをロード
    model_path = 'best_model.pth'
    if not os.path.exists(model_path):
        video_log_widget.value += f"エラー: モデルファイル '{model_path}' が見つかりません。先に学習を実行してください。"
        create_video_button.disabled = False
        return
    
    eval_model = get_model()
    eval_model.load_state_dict(torch.load(model_path))
    eval_model.eval()

    dataset_path = video_dataset_widget.value
    if not dataset_path:
        video_log_widget.value += "エラー: 評価するデータセットを選択してください。"
        create_video_button.disabled = False
        return

    image_dir = os.path.join(dataset_path, 'xy')
    if not os.path.exists(image_dir):
        video_log_widget.value += f"エラー: ディレクトリ '{image_dir}' が見つかりません。"
        create_video_button.disabled = False
        return
        
    image_files = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))
    if not image_files:
        video_log_widget.value += "エラー: 画像ファイルが見つかりません。"
        create_video_button.disabled = False
        return

    # 動画の保存先
    output_dir = 'videos'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    output_path = os.path.join(output_dir, video_name_widget.value)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, 20.0, (224, 224))
    
    video_log_widget.value += f"全 {len(image_files)} フレームの処理を開始します...\n"

    for image_path in tqdm(image_files, desc="動画作成中"):
        img = cv2.imread(image_path)
        img_resized = cv2.resize(img, (224, 224))
        with torch.no_grad():
            preprocessed_img = preprocess(img_resized).to(device)
            output = eval_model(preprocessed_img).detach().cpu().numpy().flatten()
        
        x = (output[0] / 2.0 + 0.5) * 224
        y = (output[1] / 2.0 + 0.5) * 224
        cv2.circle(img_resized, (int(x), int(y)), 8, (255, 0, 0), -1)
        out.write(img_resized)

    out.release()
    video_log_widget.value += f"\n✅ 動画の作成が完了しました: {output_path}\n"
    
    # 動画プレイヤーの更新
    with video_player_widget:
        clear_output(wait=True)
        mp4 = open(output_path,'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
        display(HTML(f'<video width=400 controls src="{data_url}"></video>'))
        
    create_video_button.disabled = False

# --- イベントリスナー (評価) ---
create_video_button.on_click(create_video)

# --- UI表示 (評価) ---
eval_ui = widgets.VBox([
    widgets.HBox([video_dataset_widget, video_name_widget]),
    create_video_button,
    video_log_widget,
    video_player_widget,
])
display(eval_ui)